In [1]:
"""
EC4430 Portfolio Management: S&P 100 constituent prices + S&P 500 market proxy
=============================================================================
Daily ADJUSTED close prices from Yahoo Finance, 1 Oct 2021 to 30 Oct 2026:
    Table 1  stocks_adj_close  - every S&P 100 constituent
    Table 2  market_adj_close  - the S&P 500 index (^GSPC), the market proxy

Works as a plain script (python sp100_sp500_data.py) or pasted into ONE
notebook cell. Missing packages are installed automatically (section 0),
including in virtual environments that were created without pip (e.g. by uv).

Outputs (saved next to the notebook/script):
    sp100_sp500_prices.xlsx  - sheets: SP100_AdjClose, SP500_AdjClose, Coverage, SP100_Members
    sp100_adj_close.csv
    sp500_adj_close.csv
"""

# =============================================================================
# 0. SETUP:
# =============================================================================
import importlib
import importlib.util
import shutil
import subprocess
import sys

# import name -> pip package name
REQUIRED_PACKAGES = {
    "yfinance": "yfinance",
    "pandas": "pandas",
    "numpy": "numpy",
    "requests": "requests",
    "lxml": "lxml",          # used by pd.read_html to parse the Wikipedia table
    "openpyxl": "openpyxl",  # used to write the .xlsx file
}


def _missing_packages() -> list[str]:
    return [pkg for mod, pkg in REQUIRED_PACKAGES.items()
            if importlib.util.find_spec(mod) is None]


def _run(cmd: list[str]) -> bool:
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[setup] '{' '.join(cmd[:4])} ...' failed:\n{result.stderr.strip()[-500:]}")
    return result.returncode == 0


def ensure_packages() -> None:
    missing = _missing_packages()
    if not missing:
        return
    print(f"[setup] Installing into {sys.executable}: {missing}")

    installed = False
    # (a) uv: works in uv-created venvs that have no pip (e.g. GitHub Codespaces templates)
    if shutil.which("uv"):
        installed = _run(["uv", "pip", "install", "--python", sys.executable, *missing])
    # (b) pip: bootstrap pip first with ensurepip if the environment lacks it
    if not installed:
        if importlib.util.find_spec("pip") is None:
            _run([sys.executable, "-m", "ensurepip", "--upgrade"])
        installed = _run([sys.executable, "-m", "pip", "install", "--quiet", *missing])

    importlib.invalidate_caches()
    still_missing = _missing_packages()
    if still_missing:
        raise ImportError(
            f"Could not install {still_missing}. In a terminal, run:\n"
            f"    uv pip install --python {sys.executable} {' '.join(still_missing)}\n"
            "then restart the kernel and run this again."
        )
    print("[setup] Done.")


ensure_packages()

# =============================================================================
# 1. IMPORTS
# =============================================================================
import io
import logging
import time

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Silence yfinance's own error spam; the script reports failures itself
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# =============================================================================
# 2. SETTINGS
# =============================================================================
START = "2021-10-01"
END_INCLUSIVE = "2026-10-30"
# yfinance treats `end` as EXCLUSIVE, so add one day to keep 30 Oct 2026
END_EXCLUSIVE = (pd.Timestamp(END_INCLUSIVE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

MARKET_TICKER = "^GSPC"          # S&P 500 price index
ADD_TOTAL_RETURN_INDEX = True   # True also pulls ^SP500TR (dividends reinvested), see NOTES

OUT_XLSX = "sp100_sp500_prices.xlsx"
OUT_STOCKS_CSV = "sp100_adj_close.csv"
OUT_MARKET_CSV = "sp500_adj_close.csv"

# Where the ticker list comes from:
#   False -> use the written list below (default)
#   True  -> scrape Wikipedia's current S&P 100 table (falls back to the list if it fails)
USE_WIKIPEDIA = False

# S&P 100 members after the 2026 index changes:
#   23 Mar 2026  out: AIG, MET, PYPL, TGT        in: AMAT, GEV, LRCX, MU
#   29-30 Jun 2026  out: HON (replaced by HONA, which was removed in Sept)
#   21 Sep 2026  out: CL, NKE, SPG               in: ANET, DELL, PANW, SNDK
# Ticker change: BNY Mellon BK -> BNY on 21 May 2026 (Yahoo now files the
# whole price history under BNY, so 'BK' returns nothing).
# GEV (listed Apr 2024) and SNDK (listed Feb 2025) don't have data back to
# Oct 2021, so they will show up in the coverage report with missing data.
SP100_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AMAT", "AMD", "AMGN", "AMT", "AMZN",
    "ANET", "AVGO", "AXP", "BA", "BAC", "BKNG", "BLK", "BMY", "BNY", "BRK-B",
    "C", "CAT", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS", "CVX",
    "DE", "DELL", "DHR", "DIS", "DUK", "EMR", "FDX", "GD", "GE",
     "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "IBM", "INTC", "INTU",
    "ISRG", "JNJ", "JPM", "KO", "LIN", "LLY", "LMT", "LOW", "LRCX", "MA",
    "MCD", "MDLZ", "MDT", "META", "MMM", "MO", "MRK", "MS", "MSFT", "MU",
    "NEE", "NFLX", "NOW", "NVDA", "ORCL", "PANW", "PEP", "PFE", "PG", "PLTR",
    "PM", "QCOM", "RTX", "SBUX", "SCHW", "SO", "T", "TMO", "TMUS",
    "TSLA", "TXN", "UBER", "UNH", "UNP", "UPS", "USB", "V", "VZ", "WFC",
    "WMT", "XOM",
]


# =============================================================================
# 3. S&P 100 TICKER LIST
# =============================================================================
def get_sp100_tickers() -> tuple[list[str], pd.DataFrame | None]:
    """Return the written SP100_TICKERS list, or scrape Wikipedia if USE_WIKIPEDIA.

    Returns (tickers, info_table). info_table has Symbol / Name / Sector when
    the scrape works, otherwise None.
    """
    if not USE_WIKIPEDIA:
        tickers = sorted(set(SP100_TICKERS))
        print(f"[tickers] Using the written S&P 100 list: {len(tickers)} tickers.")
        return tickers, None

    url = "https://en.wikipedia.org/wiki/S%26P_100"
    try:
        # Wikipedia blocks requests without a browser-like User-Agent
        html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20).text
        tables = pd.read_html(io.StringIO(html))
        info = next(t for t in tables if "Symbol" in t.columns)
        info = info.rename(columns=str.strip)
        # Yahoo uses '-' rather than '.' for share classes (BRK.B -> BRK-B)
        info["Symbol"] = info["Symbol"].str.strip().str.replace(".", "-", regex=False)
        tickers = sorted(info["Symbol"].unique().tolist())
        print(f"[tickers] Scraped {len(tickers)} S&P 100 members from Wikipedia.")
        return tickers, info
    except Exception as exc:  # noqa: BLE001
        print(f"[tickers] Wikipedia scrape failed ({exc}); using the written list.")
        return sorted(set(SP100_TICKERS)), None


# =============================================================================
# 4. DOWNLOAD ADJUSTED CLOSE PRICES
# =============================================================================
def download_adj_close(tickers: list[str], retries: int = 3) -> pd.DataFrame:
    """Download daily Adj Close for a list of tickers; returns Date x Ticker."""
    prices = pd.DataFrame()
    remaining = list(tickers)

    for attempt in range(1, retries + 1):
        raw = yf.download(
            remaining,
            start=START,
            end=END_EXCLUSIVE,
            interval="1d",
            auto_adjust=False,   # keep raw Close AND a separate 'Adj Close' column
            actions=False,
            progress=False,
            threads=True,
        )
        if raw.empty:
            adj = pd.DataFrame(columns=remaining)
        else:
            adj = raw["Adj Close"]
            if isinstance(adj, pd.Series):          # single-ticker edge case
                adj = adj.to_frame(name=remaining[0])

        got = adj.dropna(axis=1, how="all")         # keep tickers that returned data
        prices = pd.concat([prices, got], axis=1)

        remaining = [t for t in remaining if t not in prices.columns]
        if not remaining:
            break
        time.sleep(2 * attempt)  # back off; Yahoo rate-limits bursts

    if remaining:
        print(f"[download] No data for {remaining} after {retries} attempts. "
              "Check the ticker hasn't changed (e.g. BK -> BNY) or been delisted.")
    return prices


# =============================================================================
# 5. TIDY INTO NEAT TABLES
# =============================================================================
def tidy(prices: pd.DataFrame) -> pd.DataFrame:
    """Standard formatting: tz-naive DatetimeIndex called 'Date', sorted, float64."""
    out = prices.copy()
    out.index = pd.to_datetime(out.index).tz_localize(None).normalize()
    out.index.name = "Date"
    out.columns.name = None
    out = out.loc[START:END_INCLUSIVE]
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out = out.reindex(sorted(out.columns), axis=1).astype("float64")
    return out


def coverage_report(stocks: pd.DataFrame) -> pd.DataFrame:
    """One row per ticker: first/last date, number of obs, % missing."""
    n_days = len(stocks)
    rep = pd.DataFrame({
        "first_date": stocks.apply(pd.Series.first_valid_index),
        "last_date": stocks.apply(pd.Series.last_valid_index),
        "n_obs": stocks.notna().sum(),
    })
    rep["pct_missing"] = np.round(100 * (1 - rep["n_obs"] / n_days), 2)
    rep.index.name = "Ticker"
    return rep.sort_values(["pct_missing", "Ticker"], ascending=[False, True])


# =============================================================================
# 6. NEAT TABLE DISPLAY
# =============================================================================
def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None and "IPKernelApp" in get_ipython().config
    except Exception:  # noqa: BLE001
        return False


TABLE_CSS = [
    {"selector": "", "props": "border-collapse: collapse; margin-bottom: 18px;"},
    {"selector": "th", "props": "text-align: right; padding: 4px 10px; font-weight: 600;"
                                "border-bottom: 1px solid rgba(127,127,127,0.5); white-space: nowrap;"},
    {"selector": "th.row_heading", "props": "text-align: left;"},
    {"selector": "td", "props": "text-align: right; padding: 4px 10px;"
                                "font-variant-numeric: tabular-nums; white-space: nowrap;"},
    {"selector": "tbody tr:nth-child(even)", "props": "background-color: rgba(127,127,127,0.08);"},
    {"selector": "tbody tr:hover", "props": "background-color: rgba(127,127,127,0.18);"},
]


def show_table(df: pd.DataFrame, title: str, fmt="{:,.2f}") -> None:
    """Notebook: formatted HTML table (scrolls sideways). Script: aligned text."""
    out = df.copy()
    if isinstance(out.index, pd.DatetimeIndex):
        out.index = out.index.strftime("%d %b %Y")
    out.index.name = None      # avoids an extra, mostly empty header row
    out.columns.name = None
    if _in_notebook():
        from IPython.display import HTML, display
        styler = out.style.format(fmt, na_rep="–").set_table_styles(TABLE_CSS)
        display(HTML(
            f'<div style="font-weight:600; font-size:1.1em; margin:14px 0 6px">{title}</div>'
            f'<div style="overflow-x:auto; max-width:100%">{styler.to_html()}</div>'
        ))
    else:
        print(f"\n{title}\n{'-' * len(title)}")
        print(out.to_string(float_format=lambda x: f"{x:,.2f}", na_rep="-"))


def preview(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """First n and last n rows, with a '...' marker row between them."""
    gap = pd.DataFrame(np.nan, index=["…"], columns=df.columns)
    head, tail = df.head(n), df.tail(n)
    head.index, tail.index = head.index.strftime("%d %b %Y"), tail.index.strftime("%d %b %Y")
    return pd.concat([head, gap, tail])


# =============================================================================
# 7. MAIN
# =============================================================================
def main() -> tuple[pd.DataFrame, pd.DataFrame]:
    tickers, info = get_sp100_tickers()

    # ---- Table 1: S&P 100 stocks ----
    stocks_adj_close = tidy(download_adj_close(tickers))

    # ---- Table 2: S&P 500 market proxy ----
    mkt_tickers = [MARKET_TICKER] + (["^SP500TR"] if ADD_TOTAL_RETURN_INDEX else [])
    market_adj_close = tidy(download_adj_close(mkt_tickers))
    market_adj_close = market_adj_close.rename(
        columns={"^GSPC": "SP500", "^SP500TR": "SP500_TR"}
    )[["SP500"] + (["SP500_TR"] if ADD_TOTAL_RETURN_INDEX else [])]

    # Align both tables on the same trading days. Missing stock prices stay NaN
    # (no forward-filling), so gaps are visible in the coverage report.
    common = market_adj_close.index.intersection(stocks_adj_close.index)
    stocks_adj_close = stocks_adj_close.loc[common]
    market_adj_close = market_adj_close.loc[common]

    # ---- neat tables ----
    cov = coverage_report(stocks_adj_close)
    summary = pd.DataFrame({
        "Table": ["1. S&P 100 stocks", "2. S&P 500 (market proxy)"],
        "Start": [f"{stocks_adj_close.index.min():%d %b %Y}"] * 2,
        "End": [f"{stocks_adj_close.index.max():%d %b %Y}"] * 2,
        "Trading days": [len(stocks_adj_close), len(market_adj_close)],
        "Series": [stocks_adj_close.shape[1], market_adj_close.shape[1]],
    }).set_index("Table")
    show_table(summary, "Dataset summary", fmt="{}")

    show_table(preview(stocks_adj_close),
               "Table 1: S&P 100 daily adjusted close (USD), first and last 5 days")
    show_table(preview(market_adj_close),
               "Table 2: S&P 500 daily close (index points), first and last 5 days")

    gaps = cov[cov["pct_missing"] > 0].copy()
    if gaps.empty:
        print("Coverage: every stock has a full price history.")
    else:
        gaps["first_date"] = gaps["first_date"].dt.strftime("%d %b %Y")
        gaps["last_date"] = gaps["last_date"].dt.strftime("%d %b %Y")
        gaps.columns = ["First price", "Last price", "Days with data", "% missing"]
        show_table(gaps, "Stocks without a full history (listed or spun off after Oct 2021)",
                   fmt={"Days with data": "{:,.0f}", "% missing": "{:.1f}%"})

    # ---- save ----
    stocks_adj_close.to_csv(OUT_STOCKS_CSV)
    market_adj_close.to_csv(OUT_MARKET_CSV)
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl", date_format="YYYY-MM-DD",
                        datetime_format="YYYY-MM-DD") as xl:
        stocks_adj_close.to_excel(xl, sheet_name="SP100_AdjClose")
        market_adj_close.to_excel(xl, sheet_name="SP500_AdjClose")
        cov.to_excel(xl, sheet_name="Coverage")
        if info is not None:
            info.to_excel(xl, sheet_name="SP100_Members", index=False)
    print(f"Saved: {OUT_XLSX}, {OUT_STOCKS_CSV}, {OUT_MARKET_CSV}")
    print("Full tables are in memory as `stocks_adj_close` and `market_adj_close`.")

    return stocks_adj_close, market_adj_close


if __name__ == "__main__":
    # In a notebook these two tables stay in memory for the next cells
    stocks_adj_close, market_adj_close = main()

    # Next step for CAPM betas / Sharpe ratios: simple daily returns, e.g.
    # stock_rets  = stocks_adj_close.pct_change().iloc[1:]
    # market_rets = market_adj_close["SP500"].pct_change().iloc[1:]

# =============================================================================
# NOTES
# * ^GSPC is a PRICE index, so Yahoo's "Adj Close" for it equals "Close"
#   (index levels aren't adjusted for dividends). The stocks' Adj Close DOES
#   include dividends, so stock returns are total returns while market returns
#   are price-only, which biases alphas up by roughly the ~1.3-1.5% p.a.
#   dividend yield. For CAPM regressions, set ADD_TOTAL_RETURN_INDEX = True
#   and use SP500_TR.
# * Survivorship bias: the ticker list is the CURRENT membership applied back
#   to 2021. Firms that left the index (e.g. AIG, MET, PYPL, TGT, CL, NKE, SPG
#   in 2026) are excluded and recent winners that joined (MU, LRCX, ANET, SNDK)
#   are included for the whole sample, which flatters any backtest. Mention
#   this in the essay if you backtest on this universe.
# =============================================================================

[tickers] Using the written S&P 100 list: 99 tickers.


,Start,End,Trading days,Series
1. S&P 100 stocks,01 Oct 2021,30 Oct 2026,1276,99
2. S&P 500 (market proxy),01 Oct 2021,30 Oct 2026,1276,2


,AAPL,ABBV,ABT,ACN,ADBE,AMAT,AMD,AMGN,AMT,AMZN,ANET,AVGO,AXP,BA,BAC,BKNG,BLK,BMY,BNY,BRK-B,C,CAT,CMCSA,COF,COP,COST,CRM,CSCO,CVS,CVX,DE,DELL,DHR,DIS,DUK,EMR,FDX,GD,GE,GILD,GM,GOOG,GOOGL,GS,HD,IBM,INTC,INTU,ISRG,JNJ,JPM,KO,LIN,LLY,LMT,LOW,LRCX,MA,MCD,MDLZ,MDT,META,MMM,MO,MRK,MS,MSFT,MU,NEE,NFLX,NOW,NVDA,ORCL,PANW,PEP,PFE,PG,PLTR,PM,QCOM,RTX,SBUX,SCHW,SO,T,TMO,TMUS,TSLA,TXN,UBER,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
01 Oct 2021,139.03,90.25,105.70,296.67,577.47,123.90,102.45,183.10,226.65,164.16,21.91,44.61,163.18,226.00,38.12,95.96,745.73,48.38,45.97,275.71,59.68,177.89,44.86,152.50,58.71,424.24,270.10,47.76,70.71,85.38,320.62,48.64,261.27,170.65,81.21,86.20,162.26,177.61,64.32,57.38,50.92,135.18,135.26,336.63,290.60,112.38,49.61,522.03,336.50,139.56,146.74,45.80,278.36,219.40,304.26,183.73,54.16,349.15,216.22,50.84,108.50,339.76,124.00,31.23,69.86,83.96,277.46,69.31,69.25,61.31,126.56,20.65,83.58,81.08,128.36,32.48,121.95,24.33,76.34,115.45,78.57,100.30,69.61,51.44,15.00,564.60,120.52,258.41,167.38,47.05,358.52,180.24,143.27,48.77,221.48,38.98,41.38,42.91,51.13
04 Oct 2021,135.61,89.95,104.18,291.76,558.49,120.13,100.34,180.97,225.48,159.49,21.48,43.57,161.98,223.79,38.35,96.10,731.26,47.91,45.91,273.64,59.77,176.79,44.52,152.11,59.87,416.49,265.79,47.29,70.50,85.70,317.10,48.39,258.03,168.18,83.10,84.82,158.87,176.98,63.76,56.54,51.74,132.51,132.40,331.45,287.61,113.00,49.25,504.16,323.50,138.47,146.59,45.78,275.37,214.87,302.67,184.46,52.30,333.66,216.35,50.72,106.57,323.14,123.55,31.67,71.31,82.35,271.71,68.95,69.38,60.33,123.59,19.65,83.13,78.26,127.72,32.09,121.26,23.17,76.43,113.63,78.76,99.00,68.49,52.00,15.05,560.93,119.29,260.51,164.45,45.72,353.57,183.61,140.85,48.58,215.98,39.12,41.38,42.50,51.80
05 Oct 2021,137.53,90.59,104.50,296.90,566.70,122.88,101.81,181.33,224.90,161.05,22.06,44.43,163.95,224.42,39.13,95.04,742.97,48.09,47.27,278.05,60.81,176.22,44.73,154.50,60.07,422.27,266.68,47.69,70.35,86.63,316.95,48.70,258.85,169.30,82.62,86.49,163.53,177.56,63.76,56.41,52.08,134.90,134.74,341.78,290.26,112.25,49.69,511.84,330.07,138.78,148.98,45.85,278.34,220.68,304.21,184.25,52.84,332.70,218.26,51.38,107.21,329.80,124.84,31.44,70.03,84.09,277.13,68.83,69.49,63.48,126.68,20.36,85.21,79.26,128.48,32.01,121.73,23.21,76.59,114.39,79.69,98.46,70.99,51.69,15.11,563.77,119.23,260.20,167.46,45.61,359.44,187.59,141.84,49.19,215.54,39.13,42.21,42.78,51.71
06 Oct 2021,138.40,90.44,105.05,296.97,570.31,121.39,103.64,179.72,227.80,163.10,22.09,44.74,162.67,224.99,39.14,95.66,741.88,48.12,47.37,280.22,60.72,175.63,44.81,153.72,60.14,425.20,270.11,47.04,69.94,85.87,317.06,48.51,259.78,170.14,84.00,86.63,163.48,178.63,63.42,56.47,51.69,136.07,136.27,343.76,291.04,111.63,49.72,513.05,334.94,139.07,149.30,46.40,276.68,217.74,306.17,184.00,52.45,338.02,219.93,52.58,107.46,330.47,125.20,31.63,69.22,84.00,281.31,68.29,70.97,63.91,125.87,20.61,85.07,81.02,131.77,31.79,123.35,23.58,76.79,114.87,80.18,99.16,70.81,52.25,15.08,567.44,118.47,260.92,167.90,47.01,360.19,189.80,142.65,49.32,217.69,39.14,42.13,43.09,50.76
07 Oct 2021,139.65,91.72,106.33,297.94,578.96,122.70,106.45,178.99,226.23,165.12,22.76,45.21,163.91,226.48,39.04,96.47,746.13,47.98,47.36,282.95,60.53,179.19,45.21,153.73,59.89,428.54,269.44,47.98,70.82,86.48,322.39,49.22,261.52,172.30,83.27,88.17,162.57,181.84,64.13,57.02,54.10,137.88,137.91,346.04,297.31,111.20,49.90,516.70,338.34,140.31,150.24,46.54,279.59,222.83,307.74,187.52,52.84,343.51,221.02,52.31,108.09,326.10,124.78,31.65,70.32,83.75,282.98,68.87,70.75,63.19,127.58,20.98,85.98,82.28,132.98,32.33,124.34,23.73,76.00,114.67,80.97,99.67,71.45,52.20,15.35,575.23,117.12,264.54,168.69,47.91,369.73,189.30,143.36,49.67,221.57,39.15,42.03,43.60,50.91
…,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–
26 Oct 2026,343.65,286.74,104.88,247.23,258.62,387.59,531.28,444.63,188.28,249.09,177.24,32

,SP500,SP500_TR
01 Oct 2021,"4,357.04","9,098.25"
04 Oct 2021,"4,300.46","8,980.64"
05 Oct 2021,"4,345.72","9,076.19"
06 Oct 2021,"4,363.55","9,113.45"
07 Oct 2021,"4,399.76","9,191.19"
…,–,–
26 Oct 2026,"7,676.77","17,226.60"
27 Oct 2026,"7,722.60","17,329.47"
28 Oct 2026,"7,743.98","17,377.56"
29 Oct 2026,"7,720.39","17,324.68"


Coverage: every stock has a full price history.
Saved: sp100_sp500_prices.xlsx, sp100_adj_close.csv, sp500_adj_close.csv
Full tables are in memory as `stocks_adj_close` and `market_adj_close`.


In [2]:
"""
Phase 2: Returns (run in a NEW cell, after the price-download cell)
=====================================================================
R1 (volatility):  last 252 daily log returns         ln(P_t / P_t-1)
                  Excel: =LN(Prices!B1013/Prices!B1012), rows 1013-1264
R3 (correlation): overlapping 3-day log returns      ln(P_t / P_t-3)
                  Excel: =LN(Prices!B5/Prices!B2), rows 5-1264

Uses `stocks_adj_close` from the previous cell (or reloads sp100_adj_close.csv).
Saves sheets R1 and R3 to returns_r1_r3.xlsx, ticker headers included.
"""
import numpy as np
import pandas as pd

# ---- settings -------------------------------------------------------------
R1_WINDOW = 252   # trading days in the volatility window (~1 year)
R3_HORIZON = 3    # 3-day returns for the correlation sheet

# The Excel plan assumes 1,263 price rows (rows 2-1264), which gives exactly
# 1,260 R3 returns. Your download has more days than that. Leave this as None
# to use ALL the data, or set it to 1263 to match the plan's row counts exactly
# (keeps the most recent 1,263 days).
N_PRICE_ROWS = 1263

OUT_XLSX = "returns_r1_r3.xlsx"

# ---- load prices ------------------------------------------------------------
try:
    prices = stocks_adj_close.copy()          # from the previous cell
except NameError:
    prices = pd.read_csv("sp100_adj_close.csv", index_col="Date", parse_dates=True)

if N_PRICE_ROWS is not None:
    prices = prices.iloc[-N_PRICE_ROWS:]

log_p = np.log(prices)

# ---- R1: last 252 daily log returns -----------------------------------------
# diff() gives ln(P_t) - ln(P_t-1) = ln(P_t / P_t-1); keep the final 252 rows
R1 = log_p.diff().iloc[-R1_WINDOW:]

# ---- R3: overlapping 3-day log returns --------------------------------------
# diff(3) gives ln(P_t / P_t-3); the first 3 rows have no t-3 price, so drop them
R3 = log_p.diff(R3_HORIZON).iloc[R3_HORIZON:]

# ---- checks -----------------------------------------------------------------
assert len(R1) == R1_WINDOW
assert len(R3) == len(prices) - R3_HORIZON
for name, df in [("R1", R1), ("R3", R3)]:
    n_nan = int(df.isna().sum().sum())
    if n_nan:
        print(f"Warning: {name} has {n_nan} missing values "
              f"(tickers: {df.columns[df.isna().any()].tolist()})")

summary = pd.DataFrame({
    "Sheet": ["R1 (volatility)", "R3 (correlation)"],
    "Return": ["1-day log", f"{R3_HORIZON}-day log, overlapping"],
    "First date": [f"{R1.index[0]:%d %b %Y}", f"{R3.index[0]:%d %b %Y}"],
    "Last date": [f"{R1.index[-1]:%d %b %Y}", f"{R3.index[-1]:%d %b %Y}"],
    "Rows": [len(R1), len(R3)],
    "Tickers": [R1.shape[1], R3.shape[1]],
}).set_index("Sheet")

# ---- neat display (reuses show_table from the first cell if available) ------
def _pct_preview(df, n=5):
    gap = pd.DataFrame(np.nan, index=["…"], columns=df.columns)
    h, t = df.head(n).copy(), df.tail(n).copy()
    h.index, t.index = h.index.strftime("%d %b %Y"), t.index.strftime("%d %b %Y")
    return pd.concat([h, gap, t]) * 100   # show in % for readability

try:
    show_table(summary, "Phase 2: return sheets", fmt="{}")
    show_table(_pct_preview(R1), "R1: daily log returns (%), last 252 days, first and last 5 rows",
               fmt="{:.2f}")
    show_table(_pct_preview(R3), "R3: overlapping 3-day log returns (%), first and last 5 rows",
               fmt="{:.2f}")
except NameError:
    print(summary.to_string())
    print("\nR1 (%):\n", (R1.tail() * 100).round(2).to_string())
    print("\nR3 (%):\n", (R3.tail() * 100).round(2).to_string())

# ---- save (decimal log returns, full precision) -----------------------------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl", datetime_format="YYYY-MM-DD") as xl:
    R1.to_excel(xl, sheet_name="R1")
    R3.to_excel(xl, sheet_name="R3")
print(f"Saved: {OUT_XLSX}  (values are decimals, e.g. 0.0123 = 1.23%)")
print("In memory as `R1` and `R3`.")

,Return,First date,Last date,Rows,Tickers
R1 (volatility),1-day log,30 Oct 2025,30 Oct 2026,252,99
R3 (correlation),"3-day log, overlapping",25 Oct 2021,30 Oct 2026,1260,99


,AAPL,ABBV,ABT,ACN,ADBE,AMAT,AMD,AMGN,AMT,AMZN,ANET,AVGO,AXP,BA,BAC,BKNG,BLK,BMY,BNY,BRK-B,C,CAT,CMCSA,COF,COP,COST,CRM,CSCO,CVS,CVX,DE,DELL,DHR,DIS,DUK,EMR,FDX,GD,GE,GILD,GM,GOOG,GOOGL,GS,HD,IBM,INTC,INTU,ISRG,JNJ,JPM,KO,LIN,LLY,LMT,LOW,LRCX,MA,MCD,MDLZ,MDT,META,MMM,MO,MRK,MS,MSFT,MU,NEE,NFLX,NOW,NVDA,ORCL,PANW,PEP,PFE,PG,PLTR,PM,QCOM,RTX,SBUX,SCHW,SO,T,TMO,TMUS,TSLA,TXN,UBER,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
30 Oct 2025,0.63,1.35,0.19,0.60,0.41,-1.37,-3.66,0.08,0.21,-3.28,-2.24,-2.49,0.18,-6.53,0.85,0.09,0.00,6.85,0.81,0.58,1.10,-0.40,-4.33,-0.10,0.07,0.85,2.04,2.19,-4.97,-1.02,-1.72,-1.60,0.48,1.44,0.74,0.29,-0.28,0.34,-1.13,-0.05,-0.67,2.42,2.49,0.90,0.40,0.60,-2.90,0.02,-1.42,1.30,1.28,0.92,-0.49,3.74,0.90,0.56,0.21,-0.16,0.03,-0.12,-0.83,-12.03,1.10,-8.13,-0.35,0.75,-2.96,-1.16,-0.15,-1.04,2.49,-2.02,-6.92,0.51,0.95,0.00,0.54,-2.17,-0.69,-0.79,0.60,-1.22,-0.21,1.65,-1.85,-0.38,-2.05,-4.75,0.16,0.76,-3.00,1.13,-2.43,0.43,1.09,-3.16,-0.01,-0.22,-1.52
31 Oct 2025,-0.38,-4.55,-0.85,0.34,0.31,0.24,0.50,2.18,-0.26,9.15,-0.47,-1.83,0.51,0.47,0.79,-0.05,-1.40,0.98,0.01,-0.21,1.00,-1.02,1.89,0.73,0.81,-0.81,1.45,0.27,1.89,2.70,-0.82,0.62,0.15,0.70,-0.73,1.99,1.95,0.07,-0.58,1.13,0.61,-0.03,-0.10,-0.10,0.01,-0.86,-0.42,1.72,0.80,-0.10,0.54,-0.12,-2.74,2.15,0.44,-0.71,-2.23,-0.31,-1.33,-0.56,-0.30,-2.76,0.03,-1.32,-0.35,-0.16,-1.52,-0.11,-0.29,2.71,-1.66,-0.20,2.20,0.90,-0.99,1.47,0.53,3.00,-0.99,2.03,0.61,-2.78,0.12,-1.09,0.28,1.99,-0.29,3.67,1.48,0.01,-0.93,0.70,1.42,-0.34,-1.25,1.98,1.13,-1.03,-0.29
03 Nov 2025,-0.49,-2.83,0.18,-0.67,-0.84,1.96,1.37,-0.72,-1.32,3.93,-0.06,-1.93,0.25,1.74,0.21,-0.93,-1.30,-0.11,0.12,-0.39,0.97,-1.16,-3.14,0.77,-0.33,1.80,0.36,1.82,0.37,-2.36,0.07,-1.18,-0.61,-0.44,-0.51,0.33,-0.33,-0.88,-0.11,1.87,-1.27,0.81,0.90,-0.49,-0.33,-0.88,-1.23,1.29,1.84,-1.39,-0.57,-1.36,-1.47,3.83,-0.80,-1.05,2.37,-1.45,-0.69,-1.42,-0.55,-1.65,-2.62,0.51,-4.14,-0.21,-0.15,4.77,0.47,-1.69,-0.59,2.14,-1.83,-0.46,-1.72,0.04,-1.58,3.29,0.19,-0.10,-0.82,0.11,-1.46,-0.95,-0.89,-0.44,-3.26,2.55,0.00,3.28,-2.30,-0.71,-2.45,-0.13,-1.13,-0.68,0.38,0.40,-0.53
04 Nov 2025,0.37,1.84,1.57,-2.26,-0.63,-3.21,-3.77,0.13,2.08,-1.86,-2.60,-2.97,-0.31,-3.23,-0.04,-0.77,-0.84,-0.94,-0.06,2.49,-0.59,-4.12,1.69,-0.46,-0.75,1.36,-2.68,-2.90,-0.46,-0.42,1.37,-3.48,-0.11,-0.58,0.13,-1.65,-0.44,0.47,-1.44,0.78,-2.00,-2.16,-2.20,0.67,1.24,-1.28,-6.46,-2.36,-1.00,0.32,-0.03,1.01,1.39,1.15,-0.61,0.90,-3.44,1.58,0.95,-1.01,0.24,-1.64,-0.99,1.12,1.65,0.22,-0.52,-7.37,-0.11,-0.65,-2.92,-4.04,-3.83,-2.17,-0.49,-1.47,-0.58,-8.27,2.09,-4.46,-0.81,-1.69,0.74,-0.45,-0.37,0.06,1.45,-5.28,-1.31,-5.20,-0.89,0.95,-0.90,-0.41,1.00,-0.38,-0.19,0.67,0.33
05 Nov 2025,0.04,0.41,-1.01,2.01,0.05,4.54,2.48,7.52,-0.51,0.35,-8.94,1.98,1.46,-0.22,-2.06,-1.63,1.28,1.91,0.64,0.23,0.67,3.86,1.81,0.33,-0.24,-0.61,-0.69,-0.30,0.60,-0.48,1.14,-1.45,-0.76,-0.10,-0.05,-3.92,1.66,-0.16,0.30,-0.34,2.80,2.38,2.41,0.29,-2.44,1.95,3.58,-0.77,1.79,-0.46,0.78,-0.22,0.07,2.07,-2.47,-1.44,5.78,0.10,2.14,-0.11,-0.02,1.37,2.30,-0.26,0.63,0.71,-1.40,8.55,0.55,0.50,-0.90,-1.77,0.86,-0.63,-0.03,1.27,-0.94,-1.50,0.50,3.90,-0.92,4.04,-0.17,-1.43,0.49,0.42,-0.75,3.93,2.61,-2.05,-0.94,-1.61,-0.37,0.67,-0.08,0.94,-0.21,-0.79,-0.40
…,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–
26 Oct 2026,-1.10,0.38,1.13,-0.86,-1.99,2.71,2.77,1.22,-0.37,0.44,2.79,2.30,0.58,-0.02,-0.08,-1.74,0.84,0.21,1.74,1.17,0.35,0.98,-2.86,0.50,0.38,0.18,-0.39,3.45,-1.01,-0.52,-0.76,2.31,1.07,-1.90,-0.43,0.07,0.62,2.17,0.97,1.68,0.51,-0.98,-1.07,1.99,-0.14,-0.05,3.96,-1.02,-0.15,0.94,0.74,-0.01,-0.25,0.45,5.01,-0.23,3.17,-1.63,0.95,-1.31,-0.65,-1.89,-0.39,-0.93,-3.26,2.20,1.53,2.94,-0.07,-3.01,-0.49,2.36,-0.08,2.57,-0.23,-0.80,-0.69,1.06,1.20,2.08,3.33,1.86,-0.94,-0.35,-0.38,1.63,0.27,5.08,4.61,-0.86,-1.78,1.39,-0.73,0.20,-0.93,-3.56,0.32,0.33,0.72
27 Oct 2026,0.92,-0.12,-1.71,-

,AAPL,ABBV,ABT,ACN,ADBE,AMAT,AMD,AMGN,AMT,AMZN,ANET,AVGO,AXP,BA,BAC,BKNG,BLK,BMY,BNY,BRK-B,C,CAT,CMCSA,COF,COP,COST,CRM,CSCO,CVS,CVX,DE,DELL,DHR,DIS,DUK,EMR,FDX,GD,GE,GILD,GM,GOOG,GOOGL,GS,HD,IBM,INTC,INTU,ISRG,JNJ,JPM,KO,LIN,LLY,LMT,LOW,LRCX,MA,MCD,MDLZ,MDT,META,MMM,MO,MRK,MS,MSFT,MU,NEE,NFLX,NOW,NVDA,ORCL,PANW,PEP,PFE,PG,PLTR,PM,QCOM,RTX,SBUX,SCHW,SO,T,TMO,TMUS,TSLA,TXN,UBER,UNH,UNP,UPS,USB,V,VZ,WFC,WMT,XOM
25 Oct 2021,-0.42,0.08,2.62,3.03,2.28,1.32,5.00,-0.97,3.05,-2.81,-2.22,2.46,1.50,-1.54,0.89,1.20,2.39,-0.45,0.15,0.96,-1.39,-0.42,-1.16,-2.67,0.66,4.24,1.31,-1.90,1.88,0.61,2.49,0.59,-2.81,0.85,0.59,-0.74,1.76,-0.25,-0.66,1.20,0.16,-2.59,-3.10,1.67,3.57,-10.59,-11.39,5.46,1.11,0.18,0.06,-0.73,1.74,0.54,1.39,2.98,-2.42,1.14,-1.00,0.08,1.44,-3.61,-0.01,-1.00,0.73,1.23,0.23,0.72,0.44,7.18,1.13,4.70,1.29,-2.80,-1.30,0.81,0.36,4.80,-0.50,-0.17,-0.42,0.74,1.58,-1.42,-1.05,0.35,-5.05,16.87,0.41,-0.61,3.26,5.24,4.32,-0.10,1.06,-1.94,0.91,2.74,0.78
26 Oct 2021,-0.11,0.67,2.43,2.62,0.60,-1.01,2.97,-0.15,2.79,-1.73,2.03,3.76,1.94,-2.14,2.38,1.31,3.04,0.74,-1.08,1.40,0.20,-0.69,-0.50,-0.51,3.22,1.72,1.81,0.22,1.67,2.09,-0.52,-2.48,-1.23,0.41,0.28,0.46,2.12,-1.68,4.07,1.10,-1.80,-2.20,-1.83,2.43,1.25,-0.94,-14.83,5.10,2.64,1.43,1.11,0.22,1.74,2.97,-11.71,1.59,-1.17,0.31,-2.08,1.11,1.95,-7.93,0.02,-0.25,1.32,3.13,-0.21,0.44,1.96,2.32,-1.41,8.55,0.70,-4.31,1.24,1.62,2.49,2.94,-0.33,-0.84,-2.71,0.42,0.99,-1.38,-1.53,-0.01,-0.42,13.03,-2.16,-0.97,2.76,4.00,8.93,0.42,0.68,-0.58,0.59,1.31,4.90
27 Oct 2021,0.11,-0.68,0.78,0.07,-0.55,-2.81,2.03,-1.28,0.34,1.69,3.09,1.48,-4.96,-3.03,-1.12,1.39,0.25,-1.95,-2.13,-0.51,-1.50,-2.28,-3.34,-9.58,-1.12,1.47,0.88,1.92,0.65,-0.65,-0.73,-5.52,-2.63,0.08,-0.99,-0.51,0.99,-1.80,-0.19,-0.70,-6.27,5.48,6.10,-0.81,1.71,-2.14,-3.23,3.71,2.16,0.01,-2.33,1.95,1.52,2.60,-12.39,1.30,-0.56,-6.61,1.78,-0.22,-0.48,-3.89,-1.43,-1.50,0.48,0.07,4.43,1.08,0.70,-0.28,-3.24,7.32,-2.29,-2.26,0.40,-0.44,0.71,2.55,-2.10,-0.32,-2.98,-0.91,-2.91,-0.49,-1.70,0.05,-0.21,13.18,-6.20,-1.73,0.96,0.75,4.04,-4.00,-6.92,-0.57,-1.21,-0.55,1.59
28 Oct 2021,2.61,1.07,0.89,-0.03,-1.20,0.19,-0.99,-0.22,0.50,3.73,5.30,1.43,-4.32,-2.39,0.57,0.69,1.15,-1.18,0.07,-0.19,0.07,0.93,-4.11,-10.08,-1.21,0.25,1.51,1.98,1.07,-0.64,-0.86,-1.99,-0.50,-1.36,-0.34,0.99,0.24,-2.58,-0.04,-0.49,-6.29,5.17,5.93,0.02,0.08,-1.42,-2.73,2.11,3.44,-0.75,-0.34,3.28,0.40,3.33,-12.72,-0.03,2.50,-8.03,3.38,1.01,-1.87,-3.65,-1.40,-7.29,5.82,1.27,5.13,1.19,2.50,0.36,-0.17,7.38,-1.71,0.12,1.50,0.07,1.29,0.90,-0.59,1.33,-2.34,-1.33,-1.09,-0.11,-0.35,2.01,0.09,4.97,-6.36,-2.44,1.32,0.84,4.79,-2.58,-10.85,0.42,1.45,-1.08,-0.06
29 Oct 2021,0.32,4.62,0.59,0.69,1.22,3.46,-2.22,-0.82,-1.75,-0.11,1.50,-0.41,-4.04,-1.33,-0.38,-0.91,1.27,0.27,0.74,-1.33,-1.41,2.17,-4.43,-8.76,-2.75,1.39,1.54,0.29,1.08,0.34,0.34,-0.20,1.56,-1.74,-1.25,-0.09,-0.97,-0.89,-2.42,-4.30,-5.26,5.97,6.08,-1.03,0.69,-1.61,1.48,2.82,3.50,-1.75,-0.88,3.43,0.76,2.51,0.12,1.18,2.63,-6.30,3.79,-0.15,-3.44,2.43,-1.93,-8.68,6.81,0.02,6.71,0.23,-0.16,3.21,3.06,3.38,-1.08,4.53,0.27,0.41,0.10,1.40,-1.74,0.83,-0.34,-8.01,-1.25,-1.53,-0.43,3.98,-1.15,8.97,-4.33,-4.90,1.27,0.28,-2.13,-3.37,-9.05,-0.17,1.04,0.45,-2.10
…,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–,–
26 Oct 2026,-0.02,0.68,4.26,2.86,1.39,-5.36,-3.96,1.42,5.82,-5.19,-3.92,-2.59,-2.68,-2.09,-0.51,-3.36,-1.93,0.80,1.34,1.76,-2.30,-6.48,-0.14,-2.11,3.84,0.68,2.38,2.74,-0.28,2.28,0.74,-3.94,-0.38,-1.29,3.08,-0.87,0.78,3.51,-2.55,3.28,0.26,-5.20,-5.51,-1.27,1.65,0.78,-2.72,1.27,5.35,3.25,-0.34,0.02,1.39,0.66,14.13,0.84,-9.36,-1.48,1.84,0.37,1.87,-10.09,-1.12,-0.28,-6.38,-1.90,-0.07,-1.10,4.69,-1.77,3.25,-2.73,-2.41,11.73,0.56,-0.20,2.60,-1.44,2.21,0.92,3.02,3.58,-2.11,2.38,-4.96,-1.32,-4.54,1.32,4.62,-2.23,-1.52,3.11,0.75,1.09,-0.91,-1.06,0.28,2.09,3.56
27 Oct 2026,-0.32,0.42,0.57,-9.56,-8.56,5.28,9.81,0.15,1.24,

Saved: returns_r1_r3.xlsx  (values are decimals, e.g. 0.0123 = 1.23%)
In memory as `R1` and `R3`.


In [3]:
"""
Phase 3: Estimates (run in a NEW cell, after the Phase 2 returns cell)
=======================================================================
Beta estimated the Frazzini & Pedersen (2014) "Betting Against Beta" way:
volatility and correlation are estimated separately, then shrunk towards 1.

    Vol_i      = STDEV.S of the last 252 daily log returns          (R1)
    Corr_i     = CORREL of overlapping 3-day log returns with mkt   (R3)
    Raw beta_i = Corr_i * Vol_i / Vol_mkt
    Beta_i     = 0.6 * Raw beta_i + 0.4

Excel plan -> pandas:
    T1  =STDEV.S(R1!B1013:B1264)        -> market_vol   (market daily vol)
    T2  1000000                         -> GROSS_EXPOSURE
    B   =INDEX(Prices!C1264:CX1264,...) -> last price in the sample
    C   =STDEV.S(INDEX(R1!...))         -> R1.std()          (ddof=1, same as STDEV.S)
    D   =CORREL(INDEX(R3!...),R3!B...)  -> R3.corrwith(mkt)  (Pearson, same as CORREL)
    E   =D2*C2/T1                       -> raw beta
    F   =0.6*E2+0.4                     -> shrunk beta
T3 (=T2/SUM(D105:D109)) needs the portfolio rows from the next phase, so it
is not calculated here.

Market proxy: column B of the Excel plan is SPY. This uses the S&P 500 index
(^GSPC) already downloaded. For daily vol and correlation the two are almost
identical (SPY just adds dividends and a tiny fee drag).
"""
import numpy as np
import pandas as pd

# ---- parameters (column T) ---------------------------------------------------
GROSS_EXPOSURE = 1_000_000   # T2
SHRINK_WEIGHT = 0.6          # weight on the raw beta
SHRINK_TARGET = 1.0          # beta is shrunk towards 1: 0.6*raw + 0.4*1
SANITY = {"Vol": (0.01, 0.03), "Corr": (0.3, 0.9), "Beta": (0.5, 1.6)}
OUT_XLSX = "estimates_phase3.xlsx"

# ---- inputs from earlier cells ----------------------------------------------
try:
    R1, R3
except NameError as exc:
    raise NameError("Run the Phase 2 returns cell first (R1 and R3 are needed).") from exc

try:
    stock_prices = prices                      # Phase 2 (trimmed if N_PRICE_ROWS was set)
except NameError:
    stock_prices = stocks_adj_close

try:
    mkt_prices = market_adj_close["SP500_TR"]
except NameError:
    mkt_prices = pd.read_csv("sp500_adj_close.csv", index_col="Date", parse_dates=True)["SP500_TR"]

# ---- market returns built exactly like R1 and R3 (column B of each sheet) ----
log_m = np.log(mkt_prices)
mkt_R1 = log_m.diff().reindex(R1.index)        # same 252 dates as R1
mkt_R3 = log_m.diff(3).reindex(R3.index)       # same dates as R3
assert mkt_R1.notna().all() and mkt_R3.notna().all(), "Market and stock dates don't line up."

# ---- T1: market daily volatility ---------------------------------------------
market_vol = mkt_R1.std()                      # STDEV.S

# ---- columns B to F ------------------------------------------------------------
estimates = pd.DataFrame({
    "Price": stock_prices.iloc[-1],            # B: last price (row 1264)
    "Vol": R1.std(),                           # C: daily vol, 1 year
    "Corr": R3.corrwith(mkt_R3),               # D: corr with market, 3-day overlapping
})
estimates["Raw beta"] = estimates["Corr"] * estimates["Vol"] / market_vol          # E
estimates["Beta"] = SHRINK_WEIGHT * estimates["Raw beta"] + (1 - SHRINK_WEIGHT) * SHRINK_TARGET  # F
estimates.index.name = "Ticker"
estimates = estimates.sort_index()

params = pd.DataFrame({
    "Cell": ["T1", "T2"],
    "Parameter": ["Market daily volatility (S&P 500)", "Gross exposure target ($)"],
    "Value": [market_vol, GROSS_EXPOSURE],
}).set_index("Cell")

# ---- sanity checks -------------------------------------------------------------
flags = {col: estimates.index[(estimates[col] < lo) | (estimates[col] > hi)].tolist()
         for col, (lo, hi) in SANITY.items()}

summary = estimates[["Vol", "Corr", "Raw beta", "Beta"]].describe().loc[
    ["mean", "50%", "min", "max", "std"]]
summary.index = ["Mean", "Median", "Min", "Max", "Std dev"]

# ---- neat display ----------------------------------------------------------------
FMT = {"Price": "${:,.2f}", "Vol": "{:.2%}", "Corr": "{:.2f}",
       "Raw beta": "{:.2f}", "Beta": "{:.2f}"}


def _outside(col_name):
    lo, hi = SANITY.get(col_name, (-np.inf, np.inf))
    return lambda s: ["background-color: rgba(220,90,60,0.25)" if (v < lo or v > hi) else ""
                      for v in s]


def _show(df, title, fmt, highlight=False):
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display
        assert get_ipython() is not None
    except Exception:  # noqa: BLE001  (plain script: print text)
        print(f"\n{title}\n{'-' * len(title)}\n{df.to_string()}")
        return
    out = df.copy()
    out.index.name = None
    sty = out.style.format(fmt)
    if highlight:
        for col in SANITY:
            sty = sty.apply(_outside(col), subset=[col])
    try:
        sty = sty.set_table_styles(TABLE_CSS)          # from the first cell, if run
    except NameError:
        pass
    display(HTML(f'<div style="font-weight:600; font-size:1.1em; margin:14px 0 6px">{title}</div>'
                 f'<div style="overflow-x:auto; max-height:600px">{sty.to_html()}</div>'))


_show(params, "Parameters (column T)", {"Value": lambda v: f"{v:.4%}" if v < 1 else f"{v:,.0f}"})
_show(summary, "Estimates: cross-section summary",
      {"Vol": "{:.2%}", "Corr": "{:.2f}", "Raw beta": "{:.2f}", "Beta": "{:.2f}"})
_show(estimates, f"Estimates sheet: {len(estimates)} stocks "
                 "(shaded = outside the sanity range)", FMT, highlight=True)

for col, (lo, hi) in SANITY.items():
    bad = flags[col]
    rng = f"{lo:.0%}-{hi:.0%}" if col == "Vol" else f"{lo}-{hi}"
    print(f"{col:<5} outside {rng}: " + (", ".join(bad) if bad else "none"))

# ---- save -------------------------------------------------------------------------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xl:
    estimates.to_excel(xl, sheet_name="Estimates")
    params.to_excel(xl, sheet_name="Parameters")
print(f"\nSaved: {OUT_XLSX}. In memory as `estimates`, `market_vol`.")

,Parameter,Value
T1,Market daily volatility (S&P 500),0.8076%
T2,Gross exposure target ($),"1,000,000"


,Vol,Corr,Raw beta,Beta
Mean,2.13%,0.46,1.28,1.17
Median,1.86%,0.51,1.14,1.08
Min,0.92%,0.12,0.19,0.52
Max,5.28%,0.75,3.59,2.55
Std dev,0.89%,0.18,0.80,0.48


,Price,Vol,Corr,Raw beta,Beta
AAPL,$344.37,1.52%,0.68,1.29,1.17
ABBV,$294.89,1.62%,0.19,0.38,0.63
ABT,$106.20,1.73%,0.33,0.70,0.82
ACN,$258.24,3.16%,0.46,1.80,1.48
ADBE,$250.23,2.61%,0.53,1.71,1.43
AMAT,$390.09,3.82%,0.65,3.06,2.23
AMD,$504.80,4.33%,0.65,3.47,2.48
AMGN,$466.01,1.86%,0.24,0.56,0.74
AMT,$187.78,1.76%,0.31,0.67,0.80
AMZN,$240.35,2.17%,0.70,1.89,1.54


Vol   outside 1%-3%: ACN, AMAT, AMD, ANET, BRK-B, DELL, IBM, INTC, INTU, LRCX, MU, NOW, ORCL, PANW, PLTR, QCOM
Corr  outside 0.3-0.9: ABBV, AMGN, BMY, COP, CVS, CVX, DUK, GILD, JNJ, KO, LLY, LMT, MDLZ, MO, MRK, PEP, PFE, PG, PM, SO, T, TMUS, UNH, VZ, XOM
Beta  outside 0.5-1.6: AMAT, AMD, ANET, AVGO, CRM, DELL, INTC, INTU, LRCX, MU, NOW, NVDA, ORCL, PLTR, QCOM, TSLA

Saved: estimates_phase3.xlsx. In memory as `estimates`, `market_vol`.


In [4]:
"""
Phase 4: Portfolio
=========================================================================
Betting-against-correlation construction (Asness, Frazzini, Gormsen & Pedersen, 2020):
  1. Sort stocks into 5 volatility quintiles                       (G)
  2. Within each quintile, rank by correlation (1 = lowest)        (H, I, J)
  3. Long the low-correlation half, short the high-correlation
     half, with rank weights that sum to 1 in each leg             (K, L, M, N)
   4. Scale each leg to a beta of 1, so every quintile is
     beta-neutral, then hold $S of the BAC factor, S/5 per quintile (rows 105-109, T3, O)          (rows 105-109, T3, O)
  5. Convert dollars into whole shares                             (P)

Excel plan -> pandas:
    G 1+(C2>T4)+(C2>T5)+(C2>T6)+(C2>T7), where T4:T7 = PERCENTILE.INC of Vol over NYSE stocks only where is this
    H  COUNTIFS(G,G2,D,"<"&D2)+1              -> min-rank of Corr within quintile
    I  COUNTIF(G,G2)                          -> stocks in quintile
    J  H-(I+1)/2                              -> distance from middle rank
    K  MAX(-J,0)    L  MAX(J,0)               -> raw low / raw high
    M  K/SUMIFS(K,G,G2)  N  L/SUMIFS(L,G,G2)  -> leg weights (sum to 1 per quintile)
    Row 105-109 B  SUMPRODUCT((G=q)*M*F)      -> low-leg beta
                C  same with N                -> high-leg beta
                D  (1/5)*(1/B+1/C)            -> size unit
        T3 BAC notional S = $1,000,000 (paper: returns per $1 of factor; gross is about 2S)
    O  (T3/5)*(M/B_q - N/C_q)                 -> $ position (+ long, - short)
    P  ROUND(O/Price,0)                       -> shares (Excel rounding: halves away from 0)
"""
import numpy as np
import pandas as pd

N_QUINTILES = 5
COMMISSION_PER_TRADE = 10      # MarketWatch game setting
OUT_XLSX = "portfolio_phase4.xlsx"

try:
    est = estimates.copy()
    gross_target = GROSS_EXPOSURE          # T2 from Phase 3
except NameError as exc:
    raise NameError("Run the Phase 3 estimates cell first.") from exc
# Paper (Sec. 2.1): need >= 750 days of returns for correlation; GEV and SNDK don't have them
est = est.drop(index=["GEV", "SNDK"], errors="ignore")
# ---- G: volatility quintile on NYSE breakpoints ----
import time
import yfinance as yf

NYSE_CODES = {"NYQ"}            # Yahoo's code for NYSE (Nasdaq = NMS/NGM/NCM)


def get_exchanges(tickers):
    out = {}
    for t in tickers:
        try:
            out[t] = yf.Ticker(t).info.get("exchange")
        except Exception:
            out[t] = None
        time.sleep(0.2)
    return pd.Series(out, name="Exchange")


est["Exchange"] = get_exchanges(est.index.tolist())
missing_exch = est.index[est["Exchange"].isna()].tolist()
if missing_exch:
    print(f"Warning: no exchange found for {missing_exch}; treated as non-NYSE.")
est["NYSE"] = est["Exchange"].isin(NYSE_CODES)

# 1. Cutoffs from NYSE stocks ONLY (20th/40th/60th/80th percentiles)
nyse_vol = est.loc[est["NYSE"], "Vol"].dropna()
cutoffs = nyse_vol.quantile([0.2, 0.4, 0.6, 0.8]).to_numpy()   # = Excel PERCENTILE.INC

# 2. Place EVERY stock (NYSE and Nasdaq) using those cutoffs, Q1 = lowest vol
bins = np.r_[-np.inf, cutoffs, np.inf]                          # open-ended Q1 and Q5
est["Vol quintile"] = pd.cut(est["Vol"], bins=bins,
                             labels=range(1, N_QUINTILES + 1)).astype(int)

# 3. Quintiles now unequal in size; each needs >= 2 stocks to form both legs
counts = est["Vol quintile"].value_counts().reindex(range(1, 6), fill_value=0)
print(f"NYSE stocks used for breakpoints: {len(nyse_vol)} of {est['Vol'].count()}")
print("Cutoffs (daily vol): " + ", ".join(f"{c:.2%}" for c in cutoffs))
print("Stocks per quintile:", counts.to_dict())
assert (counts >= 2).all(), f"A quintile has fewer than 2 stocks: {counts.to_dict()}"

# ---- H, I, J: correlation rank inside the quintile -------------------------------
grp = est.groupby("Vol quintile")
est["Corr rank"] = grp["Corr"].rank(method="min", ascending=True).astype(int)  # COUNTIFS(<)+1
est["N in quintile"] = grp["Corr"].transform("count")
est["Dist from middle"] = est["Corr rank"] - (est["N in quintile"] + 1) / 2

# ---- K, L, M, N: rank weights -------------------------------------------------------
est["Raw low"] = (-est["Dist from middle"]).clip(lower=0)
est["Raw high"] = est["Dist from middle"].clip(lower=0)
est["Low weight"] = est["Raw low"] / est.groupby("Vol quintile")["Raw low"].transform("sum")
est["High weight"] = est["Raw high"] / est.groupby("Vol quintile")["Raw high"].transform("sum")

# ---- rows 105-109: quintile table -------------------------------------------------------
quintiles = pd.DataFrame({
    "Low-leg beta": (est["Low weight"] * est["Beta"]).groupby(est["Vol quintile"]).sum(),
    "High-leg beta": (est["High weight"] * est["Beta"]).groupby(est["Vol quintile"]).sum(),
})
quintiles["Size unit"] = (1 / N_QUINTILES) * (1 / quintiles["Low-leg beta"]
                                              + 1 / quintiles["High-leg beta"])
quintiles.index.name = "Quintile"

# ---- T3: scale S -------------------------------------------------------------------------
scale_S = 1000000

# ---- O: dollar positions, P: shares ---------------------------------------------------------
b_low = est["Vol quintile"].map(quintiles["Low-leg beta"])     # VLOOKUP(G2,A105:C109,2)
b_high = est["Vol quintile"].map(quintiles["High-leg beta"])   # VLOOKUP(G2,A105:C109,3)
est["Position $"] = (scale_S / N_QUINTILES) * (est["Low weight"] / b_low
                                                - est["High weight"] / b_high)


def excel_round(x):
    """Excel ROUND(x, 0): halves round AWAY from zero (numpy rounds halves to even)."""
    return np.sign(x) * np.floor(np.abs(x) + 0.5)


est["Shares"] = excel_round(est["Position $"] / est["Price"]).astype(int)
est["Actual $"] = est["Shares"] * est["Price"]                 # what you really trade
est["Side"] = np.select([est["Shares"] > 0, est["Shares"] < 0], ["Long", "Short"], "None")

# ---- checks -----------------------------------------------------------------------------------
quintiles["Long $"] = est["Position $"].clip(lower=0).groupby(est["Vol quintile"]).sum()
quintiles["Short $"] = est["Position $"].clip(upper=0).groupby(est["Vol quintile"]).sum()
quintiles["Net beta $"] = (est["Position $"] * est["Beta"]).groupby(est["Vol quintile"]).sum()
quintiles["Stocks"] = est.groupby("Vol quintile").size()

checks = pd.DataFrame({
    "Target": [gross_target, np.nan, 0, np.nan, np.nan, np.nan],
    "Target weights": [est["Position $"].abs().sum(), est["Position $"].sum(),
                       (est["Position $"] * est["Beta"]).sum(),
                       (est["Position $"] > 0).sum(), (est["Position $"] < 0).sum(), np.nan],
    "After rounding to shares": [est["Actual $"].abs().sum(), est["Actual $"].sum(),
                                 (est["Actual $"] * est["Beta"]).sum(),
                                 (est["Shares"] > 0).sum(), (est["Shares"] < 0).sum(),
                                 (est["Shares"] != 0).sum() * COMMISSION_PER_TRADE],
}, index=["Gross exposure ($)", "Net dollar exposure ($)", "Net beta exposure ($)",
          "Long positions", "Short positions", "Commission to open ($)"])

#assert np.isclose(est["Position $"].abs().sum(), gross_target), "Gross exposure != T2"
assert np.allclose(quintiles["Net beta $"], 0, atol=1e-6), "A quintile is not beta-neutral"

# ---- trade list for the trader (Q7 appendix) ------------------------------------------------------
trades = (est.loc[est["Shares"] != 0, ["Side", "Shares", "Price", "Actual $", "Beta",
                                       "Vol quintile"]]
          .assign(Shares=lambda d: d["Shares"].abs())
          .sort_values(["Side", "Actual $"], ascending=[True, False],
                       key=lambda c: c.abs() if c.name == "Actual $" else c))

# ---- neat display ------------------------------------------------------------------------------
def _show(df, title, fmt):
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display
        assert get_ipython() is not None
    except Exception:  # noqa: BLE001
        print(f"\n{title}\n{'-' * len(title)}\n{df.to_string()}")
        return
    out = df.copy()
    out.index.name = None
    sty = out.style.format(fmt, na_rep="–")
    try:
        sty = sty.set_table_styles(TABLE_CSS)          # from the first cell, if run
    except NameError:
        pass
    display(HTML(f'<div style="font-weight:600; font-size:1.1em; margin:14px 0 6px">{title}</div>'
                 f'<div style="overflow-x:auto; max-height:600px">{sty.to_html()}</div>'))


money = lambda v: f"-${abs(v):,.0f}" if v < 0 else f"${v:,.0f}"
_show(pd.DataFrame({"Value": [scale_S]}, index=["T3: scale S"]), "Scale", {"Value": "{:,.2f}"})
_show(quintiles, "Quintile table (rows 105-109): each quintile is beta-neutral",
      {"Low-leg beta": "{:.3f}", "High-leg beta": "{:.3f}", "Size unit": "{:.4f}",
       "Long $": money, "Short $": money, "Net beta $": lambda v: f"{abs(v):,.2f}" if abs(v) < 0.005 else f"{v:,.2f}", "Stocks": "{:d}"})
_show(checks, "Portfolio checks",
      lambda v: "–" if pd.isna(v) else (f"{v:,.0f}" if float(v).is_integer() or abs(v) >= 100
                                         else f"{v:,.2f}"))
_show(est[["Vol quintile", "Corr rank", "N in quintile", "Dist from middle", "Raw low",
           "Raw high", "Low weight", "High weight", "Position $", "Shares"]]
      .sort_values(["Vol quintile", "Corr rank"]),
      "Portfolio sheet, columns G-P (sorted by quintile, then correlation rank)",
      {"Dist from middle": "{:+.1f}", "Raw low": "{:.1f}", "Raw high": "{:.1f}",
       "Low weight": "{:.3f}", "High weight": "{:.3f}", "Position $": "{:+,.0f}",
       "Shares": "{:+,d}"})
_show(trades, f"Trade list: {len(trades)} orders for MarketWatch",
      {"Shares": "{:,d}", "Price": "${:,.2f}", "Actual $": "{:+,.0f}", "Beta": "{:.2f}"})

# ---- save ----------------------------------------------------------------------------------------
portfolio = est
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xl:
    est.to_excel(xl, sheet_name="Portfolio")
    quintiles.to_excel(xl, sheet_name="Quintiles")
    checks.to_excel(xl, sheet_name="Checks")
    trades.to_excel(xl, sheet_name="Trade list")
    pd.DataFrame({"Value": [gross_target, scale_S]},
                 index=["T2 gross exposure", "T3 scale S"]).to_excel(xl, sheet_name="Parameters")
print(f"Saved: {OUT_XLSX}. In memory as `portfolio`, `quintiles`, `trades`.")

NYSE stocks used for breakpoints: 65 of 99
Cutoffs (daily vol): 1.40%, 1.67%, 1.83%, 2.10%
Stocks per quintile: {1: 16, 2: 16, 3: 15, 4: 17, 5: 35}


,Value
T3: scale S,"1,000,000.00"


,Low-leg beta,High-leg beta,Size unit,Long $,Short $,Net beta $,Stocks
1,0.574,1.000,0.5485,"$348,392","-$200,071",0.00,16
2,0.695,1.108,0.4682,"$287,615","-$180,566",0.00,16
3,0.620,1.226,0.4855,"$322,403","-$163,103",0.00,15
4,0.734,1.389,0.4163,"$272,317","-$143,943",0.00,17
5,1.336,1.859,0.2572,"$149,683","-$107,557",0.00,35


,Target,Target weights,After rounding to shares
Gross exposure ($),"1,000,000","2,175,647","2,174,937"
Net dollar exposure ($),–,"585,170","584,224"
Net beta exposure ($),0,-0.00,-800
Long positions,–,48,48
Short positions,–,48,48
Commission to open ($),–,–,960


,Vol quintile,Corr rank,N in quintile,Dist from middle,Raw low,Raw high,Low weight,High weight,Position $,Shares
JNJ,1,1,16,-7.5,7.5,0.0,0.234,0.000,"+81,654",+299
DUK,1,2,16,-6.5,6.5,0.0,0.203,0.000,"+70,767",+556
PEP,1,3,16,-5.5,5.5,0.0,0.172,0.000,"+59,880",+436
SO,1,4,16,-4.5,4.5,0.0,0.141,0.000,"+48,993",+523
KO,1,5,16,-3.5,3.5,0.0,0.109,0.000,"+38,105",+405
PG,1,6,16,-2.5,2.5,0.0,0.078,0.000,"+27,218",+168
PFE,1,7,16,-1.5,1.5,0.0,0.047,0.000,"+16,331",+543
NEE,1,8,16,-0.5,0.5,0.0,0.016,0.000,"+5,444",+65
MCD,1,9,16,+0.5,0.0,0.5,0.000,0.016,"-3,126",-12
COST,1,10,16,+1.5,0.0,1.5,0.000,0.047,"-9,378",-9


,Side,Shares,Price,Actual $,Beta,Vol quintile
JNJ,Long,299,$273.20,"+81,687",0.52,1
MO,Long,"1,033",$78.03,"+80,605",0.55,3
DUK,Long,556,$127.39,"+70,829",0.54,1
BMY,Long,"1,106",$62.48,"+69,103",0.61,3
ABBV,Long,229,$294.89,"+67,530",0.63,2
LMT,Long,95,$636.95,"+60,510",0.58,4
PEP,Long,436,$137.34,"+59,880",0.60,1
MDLZ,Long,900,$64.94,"+58,446",0.62,2
VZ,Long,"1,287",$44.73,"+57,568",0.62,3
MRK,Long,354,$149.52,"+52,930",0.58,4


Saved: portfolio_phase4.xlsx. In memory as `portfolio`, `quintiles`, `trades`.
